# 20 — Silver: conformed, typed, cleaned

**Layer:** Silver · **Engine:** PySpark · **Write format:** Delta (overwrite, versioned)

Silver is where the supplier's vocabulary becomes *our* vocabulary. Three jobs:

1. **Rename** 79 supplier columns to the snake_case names the notebooks already use.
2. **Type** them — integers as integers, scores as integers, codes as strings.
3. **Decode the sentinels.** NHS PROMs encodes "not answered" as `9`, `*`, `999`,
   `99999`. Leaving those in place is how you end up training a model that thinks a
   patient's mobility score is 9 on a 1–3 scale.

We also apply the two cohort rules from the original analysis (drop missing pre-op OKS,
drop revision surgery) and mint a stable surrogate key.

**Sentinel rule that matters clinically:** for the twelve comorbidity flags, `9` means
*"condition absent"*, not *"missing"*. Everywhere else `9` means missing. Getting this
backwards inverts twelve features.

## 1. Configuration

In [ ]:
LAKEHOUSE     = "NHS_PROMs_LH"
BRONZE_TABLE  = "bronze.knee_provider"
SILVER_TABLE  = "silver.knee_episode"
SILVER_DIM    = "silver.provider"
SUPPORTING    = "Files/data/supporting"   # the two .xlsx reference files

spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")

from pyspark.sql import functions as F
from pyspark.sql import types as T
print("ready")

## 2. The supplier → analytical column map

This dictionary *is* the data contract. It lives in code, in git, next to the notebook —
so "why is this column called `oks_t0_pain`?" has an answer a reviewer can read.

In [ ]:
# Keys are the SANITISED supplier headers as they exist in bronze: notebook 10
# replaces Delta-forbidden characters (' ,;{}()\n\t=') with underscores. The
# original header is kept in the bronze table property `supplier_header`.
COLUMN_MAP = {
    "Provider_Code":                                   "provider_code",
    "Procedure":                                       "procedure",
    "Revision_Flag":                                   "revision_flag",
    "Year":                                            "year",
    "Age_Band":                                        "age_band",
    "Gender":                                          "gender",
    "Pre-Op_Q_Assisted":                               "t0_assisted",
    "Pre-Op_Q_Assisted_By":                            "t0_assisted_by",
    "Pre-Op_Q_Symptom_Period":                         "t0_symptom_period",
    "Pre-Op_Q_Previous_Surgery":                       "t0_previous_surgery",
    "Pre-Op_Q_Living_Arrangements":                    "t0_living_arrangements",
    "Pre-Op_Q_Disability":                             "t0_disability",
    "Heart_Disease":                                   "heart_disease",
    "High_Bp":                                         "high_bp",
    "Stroke":                                          "stroke",
    "Circulation":                                     "circulation",
    "Lung_Disease":                                    "lung_disease",
    "Diabetes":                                        "diabetes",
    "Kidney_Disease":                                  "kidney_disease",
    "Nervous_System":                                  "nervous_system",
    "Liver_Disease":                                   "liver_disease",
    "Cancer":                                          "cancer",
    "Depression":                                      "depression",
    "Arthritis":                                       "arthritis",
    "Pre-Op_Q_Mobility":                               "t0_mobility",
    "Pre-Op_Q_Self-Care":                              "t0_self_care",
    "Pre-Op_Q_Activity":                               "t0_activity",
    "Pre-Op_Q_Discomfort":                             "t0_discomfort",
    "Pre-Op_Q_Anxiety":                                "t0_anxiety",
    "Pre-Op_Q_EQ5D_Index_Profile":                     "t0_eq5d_index_profile",
    "Pre-Op_Q_EQ5D_Index":                             "t0_eq5d_index",
    "Pre-Op_Q_EQ_VAS":                                 "t0_eq_vas",
    "Post-Op_Q_EQ_VAS":                                "t1_eq_vas",
    "Post-Op_Q_EQ5D_Index":                            "t1_eq5d_index",
    "Post-Op_Q_Satisfaction":                          "t1_satisfaction",
    "Post-Op_Q_Sucess":                                "t1_success",
    "Knee_Replacement_Pre-Op_Q_Pain":                  "oks_t0_pain",
    "Knee_Replacement_Pre-Op_Q_Night_Pain":            "oks_t0_night_pain",
    "Knee_Replacement_Pre-Op_Q_Washing":               "oks_t0_washing",
    "Knee_Replacement_Pre-Op_Q_Transport":             "oks_t0_transport",
    "Knee_Replacement_Pre-Op_Q_Walking":               "oks_t0_walking",
    "Knee_Replacement_Pre-Op_Q_Standing":              "oks_t0_standing",
    "Knee_Replacement_Pre-Op_Q_Limping":               "oks_t0_limping",
    "Knee_Replacement_Pre-Op_Q_Kneeling":              "oks_t0_kneeling",
    "Knee_Replacement_Pre-Op_Q_Work":                  "oks_t0_work",
    "Knee_Replacement_Pre-Op_Q_Confidence":            "oks_t0_confidence",
    "Knee_Replacement_Pre-Op_Q_Shopping":              "oks_t0_shopping",
    "Knee_Replacement_Pre-Op_Q_Stairs":                "oks_t0_stairs",
    "Knee_Replacement_Pre-Op_Q_Score":                 "oks_t0_score",
    "Knee_Replacement_Post-Op_Q_Score":                "oks_t1_score",
}

bronze = spark.table(BRONZE_TABLE)
missing = [c for c in COLUMN_MAP if c not in bronze.columns]
assert not missing, f"Bronze is missing expected columns: {missing}"
print(f"Column map validated: {len(COLUMN_MAP)} columns.")

## 3. Rename, type, and decode sentinels

`age_band` and `gender` arrive as `*` when suppressed for disclosure control — those
become `NULL`. `t0_eq_vas` uses `999`, EQ-5D index uses `99999`. The EQ-5D-derived
columns are dropped downstream anyway (they leak post-op information), but we keep them
clean in silver so the layer stands on its own.

In [ ]:
# --- column groups -------------------------------------------------------
COMORBIDITY_COLS = [
    "heart_disease", "high_bp", "stroke", "circulation", "lung_disease", "diabetes",
    "kidney_disease", "nervous_system", "liver_disease", "cancer", "depression", "arthritis",
]
# For these, 9 == "condition absent" (a real answer), NOT missing.

OKS_ITEM_COLS = [
    "oks_t0_pain", "oks_t0_night_pain", "oks_t0_washing", "oks_t0_transport",
    "oks_t0_walking", "oks_t0_standing", "oks_t0_limping", "oks_t0_kneeling",
    "oks_t0_work", "oks_t0_confidence", "oks_t0_shopping", "oks_t0_stairs",
]

# For these, 9 == "not answered" -> NULL, to be imputed at model time (MICE).
SENTINEL_9_COLS = [
    "t0_disability", "t0_discomfort", "t0_anxiety", "t0_activity", "t0_self_care",
    "t0_mobility", "t0_living_arrangements", "t0_assisted", "t0_symptom_period",
    "t0_previous_surgery", "t0_assisted_by",
]

INT_COLS = (COMORBIDITY_COLS + OKS_ITEM_COLS + SENTINEL_9_COLS +
            ["gender", "revision_flag", "oks_t0_score", "oks_t1_score",
             "t1_satisfaction", "t1_success"])
DOUBLE_COLS = ["t0_eq5d_index", "t1_eq5d_index", "t0_eq_vas", "t1_eq_vas"]
STRING_COLS = ["provider_code", "procedure", "year", "age_band", "t0_eq5d_index_profile"]

# --- rename --------------------------------------------------------------
df = bronze.select(
    *[F.col(f"`{src}`").alias(dst) for src, dst in COLUMN_MAP.items()],
    "_source_file", "_ingested_at_utc", "_batch_id", "_row_hash",
)

# --- disclosure-control markers ('*') become NULL ------------------------
for c in ["age_band", "gender"]:
    df = df.withColumn(c, F.when(F.trim(F.col(c)).isin("*", ""), None).otherwise(F.col(c)))

# --- cast ----------------------------------------------------------------
for c in INT_COLS:
    df = df.withColumn(c, F.col(c).cast(T.IntegerType()))
for c in DOUBLE_COLS:
    df = df.withColumn(c, F.col(c).cast(T.DoubleType()))

# --- numeric sentinels ---------------------------------------------------
df = (df
      .withColumn("t0_eq_vas",     F.when(F.col("t0_eq_vas")     >= 999,   None).otherwise(F.col("t0_eq_vas")))
      .withColumn("t1_eq_vas",     F.when(F.col("t1_eq_vas")     >= 999,   None).otherwise(F.col("t1_eq_vas")))
      .withColumn("t0_eq5d_index", F.when(F.col("t0_eq5d_index") >= 99999, None).otherwise(F.col("t0_eq5d_index")))
      .withColumn("t1_eq5d_index", F.when(F.col("t1_eq5d_index") >= 99999, None).otherwise(F.col("t1_eq5d_index"))))

# --- categorical sentinel 9 -> NULL (comorbidities deliberately excluded) -
for c in SENTINEL_9_COLS:
    df = df.withColumn(c, F.when(F.col(c) == 9, None).otherwise(F.col(c)))

# --- comorbidities: NHS codes 1 = yes, 9 = no ----------------------------
for c in COMORBIDITY_COLS:
    df = df.withColumn(c, F.when(F.col(c) == 1, F.lit(1))
                          .when(F.col(c) == 9, F.lit(0))
                          .otherwise(None).cast(T.IntegerType()))

# --- binary recodes to 0/1 (NHS 1 = yes, 2 = no) -------------------------
for c in ["gender", "t0_assisted", "t0_previous_surgery", "t0_disability"]:
    df = df.withColumn(c, F.when(F.col(c) == 1, F.lit(1))
                          .when(F.col(c) == 2, F.lit(0))
                          .otherwise(None).cast(T.IntegerType()))

print("Renamed, typed, and sentinel-decoded.")

## 4. Cohort rules and the surrogate key

Two exclusions, both carried over from the original analysis and both defensible to a
clinician:

* **No pre-op OKS** → we cannot compute the outcome the model is built on.
* **Revision surgery** → a different clinical population with different expectations.

`episode_id` is a deterministic hash: the same input row always produces the same id, so
the id is stable across full reprocessing. This is the key the clinician app and the Data
Agent will join on.

In [ ]:
before = df.count()

silver_df = (
    df.filter(F.col("oks_t0_score").isNotNull())
      .filter((F.col("revision_flag") != 1) | F.col("revision_flag").isNull())
      .withColumn("episode_id",
                  F.sha2(F.concat_ws("|", "_row_hash", "provider_code", "year"), 256).substr(1, 16))
      .withColumn("_silver_built_at_utc", F.current_timestamp())
)

after = silver_df.count()
print(f"{before:,} bronze rows -> {after:,} silver rows "
      f"({after / before * 100:.2f}% retained, {before - after:,} excluded)")

assert silver_df.select("episode_id").distinct().count() == after, "episode_id is not unique"
print("episode_id unique — OK")

## 5. Write silver

`overwrite` here rather than append: silver is a *derived, reproducible* view of bronze.
Rebuilding it from scratch is the correct operation, and Delta keeps the previous version
addressable via time travel, so nothing is actually lost.

In [ ]:
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year")
    .saveAsTable(SILVER_TABLE))

spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (provider_code)")
print(f"Wrote {SILVER_TABLE}")
display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").select("version","timestamp","operation"))

## 6. Silver dimension: providers

The two reference spreadsheets (academic status, ODS registry) are small and Excel-only,
so we read them on the driver with pandas and hand the result to Spark. Spark for scale,
pandas where scale is not the problem — that judgement call recurs throughout this demo.

In [ ]:
import pandas as pd

academic = pd.read_excel(f"/lakehouse/default/{SUPPORTING}/proms_providers_academic_categories.xlsx")
academic["university_hospital"] = academic["category"].map({"academisch": 1, "niet-academisch": 0})
academic = academic[["provider_code", "university_hospital"]].drop_duplicates("provider_code")

ods = pd.read_excel(f"/lakehouse/default/{SUPPORTING}/ODS Advanced Search 2.xlsx")
ods["independent_hospital"] = ods["Primary Role Name"].map({
    "INDEPENDENT SECTOR H/C PROVIDER SITE":  1,
    "INDEPENDENT SECTOR HEALTHCARE PROVIDER": 1,
    "NHS TRUST":                              0,
})
ods["provider_code"] = ods["Dataset"]
ods = ods[["provider_code", "independent_hospital", "Region"]].drop_duplicates("provider_code")

provider_pdf = (academic.merge(ods, on="provider_code", how="outer")
                        .rename(columns={"Region": "region"}))
provider_pdf["provider_code"] = provider_pdf["provider_code"].astype(str)

(spark.createDataFrame(provider_pdf)
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(SILVER_DIM))

print(f"Wrote {SILVER_DIM}: {len(provider_pdf):,} providers")
display(spark.table(SILVER_DIM).limit(10))

## 7. Silver quality report

In [ ]:
s = spark.table(SILVER_TABLE)
total = s.count()

null_report = s.select([
    (F.sum(F.col(c).isNull().cast("int")) / F.lit(total) * 100).alias(c)
    for c in ["age_band", "gender", "t0_symptom_period", "t0_living_arrangements",
              "t0_mobility", "t0_self_care", "t0_activity", "t0_discomfort",
              "t0_anxiety", "t0_disability", "t0_assisted", "t0_previous_surgery"]
])

pdf = null_report.toPandas().T.rename(columns={0: "pct_missing"}).sort_values("pct_missing", ascending=False)
print(f"Silver rows: {total:,}\n")
print(pdf.round(2).to_string())
print("\nThese are the columns notebook 40 imputes with MICE — inside the training fold only.")

---

**Next:** `30_gold_features` — join the provider dimension, engineer the clinical
features, derive the target, and publish the model-ready gold table.